<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Automatización real de archivos y directorios**

## *Inventario → clasificación → renombrado → backup → movimiento → validación*

En este laboratorio construiremos una automatización documental completa sobre un entorno de prueba real.

El objetivo es pasar de operaciones aisladas a un **pipeline reproducible y seguro**.

Seguiremos el patrón:

> **crear escenario → inventariar → detectar problemas → planificar → validar plan → crear backup → ejecutar → validar resultado → medir**

## Objetivos

Al finalizar deberías poder:

1. Crear un entorno documental de prueba.
2. Generar un inventario completo.
3. Detectar extensiones inconsistentes.
4. Detectar nombres problemáticos.
5. Detectar duplicados y colisiones.
6. Definir reglas de clasificación.
7. Normalizar nombres sin ejecutar todavía.
8. Crear un plan de operaciones.
9. Validar el plan antes de mover archivos.
10. Crear backup automático.
11. Mover archivos de forma segura.
12. Evitar sobrescrituras.
13. Registrar resultados.
14. Validar que no se pierden archivos.
15. Calcular KPIs del proceso.
16. Preparar el futuro Organizador Inteligente de Documentos.

## 1. Preparación

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import shutil
import random
import re
import time

import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)

print("Entorno preparado.")

## 2. Crear entorno documental realista

In [ ]:
raiz = Path("demo_organizador_documental")

if raiz.exists():
    shutil.rmtree(raiz)

entrada = raiz / "entrada"
salida = raiz / "salida"
backup = raiz / "backup"
logs = raiz / "logs"

for carpeta in [
    entrada,
    salida,
    backup,
    logs
]:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Entorno creado:",
    raiz.resolve()
)

## 2.1 Crear subcarpetas y documentos

In [ ]:
subcarpetas = [
    entrada / "pendientes",
    entrada / "clientes",
    entrada / "misc"
]

for carpeta in subcarpetas:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

archivos_demo = {
    entrada / "Factura 001.PDF": "contenido factura 001",
    entrada / "factura_002.pdf": "contenido factura 002",
    entrada / "ventas agosto.CSV": "ventas,100",
    entrada / "clientes.xlsx": "excel demo",
    entrada / "logo.PNG": "png demo",
    entrada / "README": "sin extension",
    entrada / "backup.zip": "zip demo",
    entrada / "documento.tmp": "tmp demo",
    entrada / "pendientes" / "factura_003.pdf": "contenido factura 003",
    entrada / "pendientes" / "presupuesto FINAL.xlsx": "excel presupuesto",
    entrada / "clientes" / "cliente A.txt": "cliente a",
    entrada / "clientes" / "cliente B.TXT": "cliente b",
    entrada / "misc" / "foto evento.jpg": "foto demo",
    entrada / "misc" / "datos.csv": "datos",
    entrada / "misc" / "archivo": "sin extension"
}

for ruta, contenido in archivos_demo.items():
    ruta.write_text(
        contenido,
        encoding="utf-8"
    )

# Crear dos archivos con mismo nombre en carpetas distintas para provocar colisión futura.
(entrada / "clientes" / "informe.pdf").write_text(
    "informe clientes",
    encoding="utf-8"
)

(entrada / "pendientes" / "informe.pdf").write_text(
    "informe pendientes",
    encoding="utf-8"
)

print(
    "Archivos creados:",
    len(
        list(
            entrada.rglob("*")
        )
    )
)

## 2.2 Visualizar árbol inicial

In [ ]:
for ruta in sorted(
    entrada.rglob("*")
):
    nivel = (
        len(
            ruta.relative_to(
                entrada
            ).parts
        )
        - 1
    )

    sangria = (
        "    " * nivel
    )

    simbolo = (
        "📁"
        if ruta.is_dir()
        else "📄"
    )

    print(
        f"{sangria}{simbolo} {ruta.name}"
    )

### Interpretación

El escenario contiene deliberadamente:

- extensiones en mayúsculas;
- espacios;
- archivos sin extensión;
- extensiones desconocidas;
- subcarpetas;
- nombres repetidos en rutas distintas.

Todavía no corregimos nada.

## 3. Crear inventario completo

In [ ]:
def hash_archivo(
    ruta,
    bloques=8192
):
    hasher = hashlib.sha256()

    with Path(
        ruta
    ).open(
        "rb"
    ) as fichero:

        while True:
            bloque = fichero.read(
                bloques
            )

            if not bloque:
                break

            hasher.update(
                bloque
            )

    return hasher.hexdigest()


def inspeccionar_archivo(
    ruta
):
    ruta = Path(
        ruta
    )

    stat = ruta.stat()

    return {
        "ruta":
            str(
                ruta
            ),

        "ruta_relativa":
            str(
                ruta.relative_to(
                    entrada
                )
            ),

        "nombre":
            ruta.name,

        "stem":
            ruta.stem,

        "extension_original":
            (
                ruta.suffix
                if ruta.suffix
                else "sin_extension"
            ),

        "extension_normalizada":
            (
                ruta.suffix.lower()
                if ruta.suffix
                else "sin_extension"
            ),

        "bytes":
            stat.st_size,

        "sha256":
            hash_archivo(
                ruta
            )
    }


inventario = pd.DataFrame(
    [
        inspeccionar_archivo(
            ruta
        )
        for ruta
        in entrada.rglob("*")
        if ruta.is_file()
    ]
)

inventario

## 3.1 Validaciones del inventario

In [ ]:
assert len(
    inventario
) == len(
    [
        ruta
        for ruta
        in entrada.rglob("*")
        if ruta.is_file()
    ]
)

assert inventario[
    "ruta"
].notna().all()

assert inventario[
    "sha256"
].str.len().eq(
    64
).all()

print(
    "Inventario validado."
)

## 4. Detectar extensiones inconsistentes

In [ ]:
conteo_original = (
    inventario[
        "extension_original"
    ]
    .value_counts()
)

conteo_original

## 4.1 Visualizar antes de normalizar

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    conteo_original.index.astype(
        str
    ),
    conteo_original.values
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Extensiones originales"
)

plt.tight_layout()
plt.show()

## 4.2 Comparar original y normalizada

In [ ]:
inventario[
    [
        "nombre",
        "extension_original",
        "extension_normalizada"
    ]
]

### Interpretación

La normalización elimina diferencias puramente sintácticas como `.PDF` frente a `.pdf`.

## 5. Detectar nombres problemáticos

In [ ]:
def detectar_problemas_nombre(
    nombre
):
    problemas = []

    if " " in nombre:
        problemas.append(
            "espacios"
        )

    if any(
        caracter.isupper()
        for caracter
        in nombre
    ):
        problemas.append(
            "mayusculas"
        )

    if not Path(
        nombre
    ).suffix:
        problemas.append(
            "sin_extension"
        )

    return problemas


inventario[
    "problemas_nombre"
] = inventario[
    "nombre"
].apply(
    detectar_problemas_nombre
)

inventario[
    [
        "nombre",
        "problemas_nombre"
    ]
]

## 5.1 Contar problemas

In [ ]:
problemas = []

for lista in inventario[
    "problemas_nombre"
]:
    problemas.extend(
        lista
    )

Counter(
    problemas
)

## 5.2 Visualizar

In [ ]:
conteo_problemas = pd.Series(
    problemas
).value_counts()

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    conteo_problemas.index,
    conteo_problemas.values
)

plt.ylabel(
    "Casos"
)

plt.title(
    "Problemas detectados en nombres"
)

plt.tight_layout()
plt.show()

## 6. Detectar duplicados reales por contenido

In [ ]:
duplicados_hash = (
    inventario.groupby(
        "sha256"
    )
    .filter(
        lambda grupo:
            len(
                grupo
            ) > 1
    )
    .sort_values(
        "sha256"
    )
)

duplicados_hash[
    [
        "ruta_relativa",
        "sha256"
    ]
]

### Interpretación

Dos archivos con el mismo nombre no son necesariamente duplicados.

Y dos archivos con nombres distintos sí podrían tener el mismo contenido.

Por eso diferenciamos:

- colisión de nombre;
- duplicado por contenido.

## 7. Detectar colisiones potenciales de nombre

In [ ]:
conteo_nombres = (
    inventario[
        "nombre"
    ]
    .str.lower()
    .value_counts()
)

nombres_colision = (
    conteo_nombres[
        conteo_nombres > 1
    ]
)

nombres_colision

## 7.1 Ver rutas implicadas

In [ ]:
inventario[
    inventario[
        "nombre"
    ]
    .str.lower()
    .isin(
        nombres_colision.index
    )
][
    [
        "ruta_relativa",
        "nombre"
    ]
]

### Interpretación

Los dos `informe.pdf` pueden terminar en la misma carpeta de destino.

Si movemos sin comprobar, uno podría sobrescribir al otro.

## 8. Definir reglas de clasificación

In [ ]:
REGLAS = {
    ".pdf": "PDF",
    ".xlsx": "Excel",
    ".csv": "Datos",
    ".txt": "Texto",
    ".jpg": "Imagenes",
    ".jpeg": "Imagenes",
    ".png": "Imagenes"
}


def clasificar_extension(
    extension
):
    if extension == "sin_extension":
        return "Revision"

    return REGLAS.get(
        extension,
        "Revision"
    )


inventario[
    "categoria"
] = inventario[
    "extension_normalizada"
].apply(
    clasificar_extension
)

inventario[
    [
        "ruta_relativa",
        "extension_normalizada",
        "categoria"
    ]
]

## 8.1 Visualizar categorías

In [ ]:
conteo_categoria = (
    inventario[
        "categoria"
    ]
    .value_counts()
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    conteo_categoria.index,
    conteo_categoria.values
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Clasificación propuesta"
)

plt.tight_layout()
plt.show()

## 9. Diseñar normalización de nombres

In [ ]:
def normalizar_nombre(
    nombre
):
    ruta = Path(
        nombre
    )

    extension = (
        ruta.suffix.lower()
        if ruta.suffix
        else ""
    )

    stem = (
        ruta.stem
        .strip()
        .lower()
    )

    stem = re.sub(
        r"\s+",
        "_",
        stem
    )

    stem = re.sub(
        r"[^a-z0-9_-]+",
        "",
        stem
    )

    return (
        f"{stem}{extension}"
        if extension
        else stem
    )


inventario[
    "nombre_normalizado"
] = inventario[
    "nombre"
].apply(
    normalizar_nombre
)

inventario[
    [
        "nombre",
        "nombre_normalizado"
    ]
]

### Importante

Todavía no renombramos físicamente.

Estamos construyendo un **plan de transformación**.

## 10. Crear estructura de salida

In [ ]:
for categoria in sorted(
    set(
        inventario[
            "categoria"
        ]
    )
):
    (
        salida
        / categoria
    ).mkdir(
        parents=True,
        exist_ok=True
    )

print(
    [
        ruta.name
        for ruta
        in salida.iterdir()
    ]
)

## 11. Generar plan de operaciones

In [ ]:
plan = inventario[
    [
        "ruta",
        "ruta_relativa",
        "nombre",
        "nombre_normalizado",
        "categoria",
        "sha256"
    ]
].copy()

plan[
    "destino_propuesto"
] = plan.apply(
    lambda fila:
        str(
            salida
            / fila[
                "categoria"
            ]
            / fila[
                "nombre_normalizado"
            ]
        ),
    axis=1
)

plan.head()

## 11.1 Detectar colisiones dentro del plan

In [ ]:
plan[
    "destino_normalizado"
] = (
    plan[
        "destino_propuesto"
    ]
    .str.lower()
)

colisiones_plan = (
    plan.groupby(
        "destino_normalizado"
    )
    .filter(
        lambda grupo:
            len(
                grupo
            ) > 1
    )
)

colisiones_plan[
    [
        "ruta_relativa",
        "destino_propuesto"
    ]
]

### Interpretación

Ahora detectamos la colisión **antes** de tocar los archivos.

Esta es una práctica clave en automatización segura.

## 12. Resolver colisiones en el plan

In [ ]:
def destino_unico_en_plan(
    destino,
    usados
):
    destino = Path(
        destino
    )

    candidato = destino
    contador = 1

    while str(
        candidato
    ).lower() in usados:

        candidato = (
            destino
            .with_name(
                f"{destino.stem}_{contador}{destino.suffix}"
            )
        )

        contador += 1

    usados.add(
        str(
            candidato
        ).lower()
    )

    return candidato


usados = set()
destinos_finales = []

for destino in plan[
    "destino_propuesto"
]:
    destinos_finales.append(
        str(
            destino_unico_en_plan(
                destino,
                usados
            )
        )
    )

plan[
    "destino_final"
] = destinos_finales

plan[
    [
        "ruta_relativa",
        "destino_final"
    ]
]

## 12.1 Validar ausencia de colisiones

In [ ]:
assert (
    plan[
        "destino_final"
    ]
    .str.lower()
    .nunique()
    == len(
        plan
    )
)

print(
    "No quedan colisiones en el plan."
)

## 13. Marcar qué archivos irán a revisión

In [ ]:
plan[
    "requiere_revision"
] = (
    plan[
        "categoria"
    ]
    == "Revision"
)

plan[
    [
        "ruta_relativa",
        "categoria",
        "requiere_revision"
    ]
]

## 14. Resumen del plan antes de ejecutar

In [ ]:
resumen_plan = pd.DataFrame({
    "metrica": [
        "Archivos totales",
        "Automáticos",
        "Revisión",
        "Destinos únicos"
    ],
    "valor": [
        len(
            plan
        ),
        int(
            (
                ~plan[
                    "requiere_revision"
                ]
            ).sum()
        ),
        int(
            plan[
                "requiere_revision"
            ].sum()
        ),
        plan[
            "destino_final"
        ].nunique()
    ]
})

resumen_plan

## 14.1 Visualizar automático vs revisión

In [ ]:
estado_plan = pd.Series({
    "Automatico":
        int(
            (
                ~plan[
                    "requiere_revision"
                ]
            ).sum()
        ),

    "Revision":
        int(
            plan[
                "requiere_revision"
            ].sum()
        )
})

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    estado_plan.index,
    estado_plan.values
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Plan antes de ejecutar"
)

plt.tight_layout()
plt.show()

## 15. Crear backup antes de mover

In [ ]:
timestamp_demo = "2026_01_01_120000"

ruta_backup_base = (
    backup
    / f"entrada_{timestamp_demo}"
)

archivo_backup = shutil.make_archive(
    base_name=str(
        ruta_backup_base
    ),
    format="zip",
    root_dir=str(
        entrada
    )
)

ruta_backup = Path(
    archivo_backup
)

print(
    "Backup:",
    ruta_backup
)

print(
    "Existe:",
    ruta_backup.exists()
)

print(
    "Bytes:",
    ruta_backup.stat().st_size
)

### Regla de seguridad

Antes de una operación masiva de movimiento o renombrado, generamos una copia recuperable.

## 16. Ejecutar movimiento

In [ ]:
def ejecutar_movimiento(
    tabla_plan
):
    registros = []

    for _, fila in tabla_plan.iterrows():

        origen = Path(
            fila[
                "ruta"
            ]
        )

        destino = Path(
            fila[
                "destino_final"
            ]
        )

        destino.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        inicio = time.perf_counter()

        shutil.move(
            str(
                origen
            ),
            str(
                destino
            )
        )

        duracion = (
            time.perf_counter()
            - inicio
        )

        registros.append({
            "origen":
                str(
                    origen
                ),

            "destino":
                str(
                    destino
                ),

            "categoria":
                fila[
                    "categoria"
                ],

            "requiere_revision":
                fila[
                    "requiere_revision"
                ],

            "estado":
                "movido",

            "segundos":
                duracion
        })

    return pd.DataFrame(
        registros
    )


resultado_movimiento = ejecutar_movimiento(
    plan
)

resultado_movimiento.head()

## 16.1 Validar que los destinos existen

In [ ]:
resultado_movimiento[
    "destino_existe"
] = resultado_movimiento[
    "destino"
].apply(
    lambda x:
        Path(
            x
        ).exists()
)

resultado_movimiento[
    "origen_existe"
] = resultado_movimiento[
    "origen"
].apply(
    lambda x:
        Path(
            x
        ).exists()
)

resultado_movimiento[
    [
        "destino_existe",
        "origen_existe"
    ]
].value_counts()

### Interpretación

Después de un movimiento correcto esperamos:

- destino existente;
- origen inexistente.

## 16.2 Validar todos los movimientos

In [ ]:
assert resultado_movimiento[
    "destino_existe"
].all()

assert (
    ~resultado_movimiento[
        "origen_existe"
    ]
).all()

print(
    "Todos los movimientos validados."
)

## 17. Validar número total de archivos

In [ ]:
archivos_salida = [
    ruta
    for ruta
    in salida.rglob("*")
    if ruta.is_file()
]

print(
    "Inventario inicial:",
    len(
        inventario
    )
)

print(
    "Archivos en salida:",
    len(
        archivos_salida
    )
)

assert (
    len(
        inventario
    )
    == len(
        archivos_salida
    )
)

## 18. Ver árbol final

In [ ]:
for ruta in sorted(
    salida.rglob("*")
):
    nivel = (
        len(
            ruta.relative_to(
                salida
            ).parts
        )
        - 1
    )

    sangria = (
        "    " * nivel
    )

    simbolo = (
        "📁"
        if ruta.is_dir()
        else "📄"
    )

    print(
        f"{sangria}{simbolo} {ruta.name}"
    )

## 19. Ver qué queda en entrada

In [ ]:
restantes = [
    ruta
    for ruta
    in entrada.rglob("*")
    if ruta.is_file()
]

restantes

### Interpretación

No queda ningún archivo en la entrada porque el plan incluía todos los documentos.

## 20. Validar hashes después del movimiento

In [ ]:
resultado_hashes = []

for _, fila in plan.iterrows():

    destino = Path(
        fila[
            "destino_final"
        ]
    )

    hash_despues = hash_archivo(
        destino
    )

    resultado_hashes.append({
        "destino":
            str(
                destino
            ),

        "hash_antes":
            fila[
                "sha256"
            ],

        "hash_despues":
            hash_despues,

        "igual":
            (
                fila[
                    "sha256"
                ]
                == hash_despues
            )
    })


validacion_hash = pd.DataFrame(
    resultado_hashes
)

validacion_hash[
    "igual"
].value_counts()

### Interpretación

El hash permite comprobar que el contenido no cambió durante el movimiento.

## 20.1 Validar integridad total

In [ ]:
assert validacion_hash[
    "igual"
].all()

print(
    "Integridad de contenidos validada."
)

## 21. Crear registro CSV del proceso

In [ ]:
ruta_registro = (
    logs
    / "resultado_movimiento.csv"
)

resultado_movimiento.to_csv(
    ruta_registro,
    index=False
)

print(
    ruta_registro.exists()
)

display(
    pd.read_csv(
        ruta_registro
    ).head()
)

## 22. KPIs del proceso

In [ ]:
kpis = {
    "archivos_procesados":
        len(
            resultado_movimiento
        ),

    "revision":
        int(
            resultado_movimiento[
                "requiere_revision"
            ].sum()
        ),

    "automaticos":
        int(
            (
                ~resultado_movimiento[
                    "requiere_revision"
                ]
            ).sum()
        ),

    "tiempo_total_seg":
        resultado_movimiento[
            "segundos"
        ].sum()
}

kpis[
    "tasa_automatizacion"
] = (
    kpis[
        "automaticos"
    ]
    / kpis[
        "archivos_procesados"
    ]
)

kpis

## 22.1 Visualizar distribución final

In [ ]:
conteo_final = (
    resultado_movimiento[
        "categoria"
    ]
    .value_counts()
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    conteo_final.index,
    conteo_final.values
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Distribución final por categoría"
)

plt.tight_layout()
plt.show()

## 23. Comparar nombre original y final

In [ ]:
comparacion_nombres = plan[
    [
        "nombre",
        "nombre_normalizado",
        "categoria",
        "destino_final"
    ]
].copy()

comparacion_nombres

## 24. Qué archivos requieren revisión

In [ ]:
revision_final = (
    plan[
        plan[
            "requiere_revision"
        ]
    ][
        [
            "ruta_relativa",
            "nombre",
            "extension_normalizada",
            "destino_final"
        ]
    ]
)

revision_final

### Interpretación

La automatización no intenta “adivinar” con reglas que no existen.

Los casos desconocidos terminan en `Revision`.

Esta será precisamente una de las zonas donde introduciremos IA posteriormente.

## 25. Simular una segunda ejecución

In [ ]:
nuevos = {
    entrada / "Factura 004.PDF":
        "factura 004",

    entrada / "datos nuevos.csv":
        "datos nuevos",

    entrada / "informe.pdf":
        "nuevo informe"
}

for ruta, contenido in nuevos.items():
    ruta.write_text(
        contenido,
        encoding="utf-8"
    )

print(
    [
        ruta.name
        for ruta
        in entrada.iterdir()
        if ruta.is_file()
    ]
)

### Nuevo problema

Una segunda ejecución puede encontrar nombres que ya existen en salida.

Por ello, el control de colisiones debe formar parte del pipeline permanentemente.

## 26. Función reusable para destino único

In [ ]:
def generar_destino_unico(
    destino
):
    destino = Path(
        destino
    )

    if not destino.exists():
        return destino

    contador = 1

    while True:
        candidato = (
            destino
            .with_name(
                f"{destino.stem}_{contador}{destino.suffix}"
            )
        )

        if not candidato.exists():
            return candidato

        contador += 1


prueba_destino = (
    salida
    / "PDF"
    / "informe.pdf"
)

print(
    "Propuesto:",
    prueba_destino
)

print(
    "Seguro:",
    generar_destino_unico(
        prueba_destino
    )
)

## 27. Diseñar pipeline final reusable

In [ ]:
def construir_plan(
    carpeta_entrada,
    carpeta_salida,
    reglas
):
    filas = []

    for ruta in Path(
        carpeta_entrada
    ).rglob("*"):

        if not ruta.is_file():
            continue

        extension = (
            ruta.suffix.lower()
            if ruta.suffix
            else "sin_extension"
        )

        categoria = (
            "Revision"
            if extension
            == "sin_extension"
            else reglas.get(
                extension,
                "Revision"
            )
        )

        nombre_normalizado = (
            normalizar_nombre(
                ruta.name
            )
        )

        destino = (
            Path(
                carpeta_salida
            )
            / categoria
            / nombre_normalizado
        )

        destino = (
            generar_destino_unico(
                destino
            )
        )

        filas.append({
            "origen":
                str(
                    ruta
                ),

            "categoria":
                categoria,

            "destino":
                str(
                    destino
                )
        })

    return pd.DataFrame(
        filas
    )


nuevo_plan = construir_plan(
    entrada,
    salida,
    REGLAS
)

nuevo_plan

## 28. Validar plan reusable

In [ ]:
assert nuevo_plan[
    "destino"
].nunique() == len(
    nuevo_plan
)

assert nuevo_plan[
    "origen"
].notna().all()

print(
    "Nuevo plan válido."
)

## 29. Mini-práctica integrada

Crea una bandeja de entrada con al menos 50 documentos y ejecuta un pipeline que:

1. inventarie;
2. calcule hashes;
3. detecte extensiones;
4. detecte problemas de nombre;
5. detecte colisiones;
6. clasifique;
7. normalice nombres;
8. cree plan;
9. valide plan;
10. cree backup ZIP;
11. mueva;
12. valide existencia;
13. valide hashes;
14. genere CSV de auditoría;
15. calcule KPIs;
16. deje casos desconocidos en `Revision`.

## 30. Checklist de automatización documental

- [ ] ¿He inventariado antes de mover?
- [ ] ¿Conozco el número inicial de archivos?
- [ ] ¿He normalizado extensiones?
- [ ] ¿He detectado nombres problemáticos?
- [ ] ¿He distinguido duplicado y colisión?
- [ ] ¿He creado un plan?
- [ ] ¿El plan tiene destinos únicos?
- [ ] ¿He creado backup?
- [ ] ¿Los casos desconocidos van a revisión?
- [ ] ¿He evitado sobrescritura?
- [ ] ¿He validado origen/destino?
- [ ] ¿He validado integridad con hash?
- [ ] ¿He generado auditoría?
- [ ] ¿He calculado KPIs?
- [ ] ¿La segunda ejecución sigue siendo segura?

## 31. Ejercicios finales

### Ejercicio 1 — Escenario
Genera 30 archivos con extensiones variadas.

### Ejercicio 2 — Inventario
Construye nombre, ruta, extensión, bytes y hash.

### Ejercicio 3 — Problemas
Detecta espacios, mayúsculas y archivos sin extensión.

### Ejercicio 4 — Hash
Crea dos archivos idénticos y detecta duplicados reales.

### Ejercicio 5 — Colisión
Crea dos archivos con el mismo nombre en carpetas diferentes.

### Ejercicio 6 — Reglas
Define clasificación por extensión.

### Ejercicio 7 — Normalización
Convierte nombres a minúsculas y sustituye espacios.

### Ejercicio 8 — Plan
Crea destino sin ejecutar movimientos.

### Ejercicio 9 — Colisiones del plan
Detecta destinos repetidos.

### Ejercicio 10 — Resolver colisiones
Añade sufijos `_1`, `_2`, etc.

### Ejercicio 11 — Backup
Crea un ZIP antes de modificar.

### Ejercicio 12 — Movimiento
Ejecuta el plan.

### Ejercicio 13 — Validación
Comprueba origen inexistente y destino existente.

### Ejercicio 14 — Integridad
Valida hash antes y después.

### Ejercicio 15 — Auditoría
Guarda resultado en CSV.

### Ejercicio 16 — KPI
Calcula tasa de automatización y revisión.

### Ejercicio 17 — Segunda ejecución
Añade nuevos documentos y evita sobrescrituras.

### Ejercicio 18 — Reto
Construye una función reusable que organice una carpeta completa de manera segura.

## 32. Preguntas de reflexión

1. ¿Por qué inventariar antes de mover?
2. ¿Qué diferencia hay entre duplicado y colisión?
3. ¿Qué aporta un hash?
4. ¿Por qué normalizar nombres?
5. ¿Qué riesgo tiene sobrescribir?
6. ¿Por qué crear un plan previo?
7. ¿Qué aporta un backup?
8. ¿Por qué validar número de archivos?
9. ¿Qué significa integridad de contenido?
10. ¿Qué información debería contener una auditoría?
11. ¿Por qué dejar desconocidos en revisión?
12. ¿Qué debe ocurrir en una segunda ejecución?
13. ¿Qué parte de este pipeline será mejorada con logging?
14. ¿Qué parte podría mejorarse con IA?
15. ¿Cómo sabrías si la automatización es fiable?

## 33. Cierre

Ya tenemos una automatización documental completa y segura:

```text
entrada
   ↓
inventario
   ↓
detección de problemas
   ↓
normalización
   ↓
clasificación
   ↓
plan
   ↓
backup
   ↓
movimiento
   ↓
validación
   ↓
auditoría
```

El siguiente laboratorio añadirá la pieza que falta para convertir esto en un sistema realmente operable:

```text
logging
+
captura de errores
+
continuación segura
+
Organizador Inteligente de Documentos
```